# Colab training command log and resume walkthrough

This notebook collects the commands used in the November 2026 text-to-SQL training sessions. Run this notebook **on the Mac**, with a Python kernel that can access the authenticated `colab` CLI. Python bodies sent through `remote()` run on Colab, not on the Mac. Do not run this controller notebook directly in Colab.

All external operations are disabled by default. Read the relevant section and enable only the operation you need; do not enable every switch and use Run All. Existing training must not be launched twice. Drive authorization remains interactive; run the mount command in a Mac terminal.

Sources: user-pasted CLI output in this conversation, `../docs/experiments.md`, `../docs/evaluation.md`, and repository training/checkpoint code. Historical observations below are transcribed records, not live notebook outputs. No authentication URLs or credentials are retained.

## Recorded experiments

| Run | Recorded result |
|---|---|
| Smoke: `qlora-r16-lr2e-4-t4-b2-smoke` | Completed; 195 training examples, 3 epochs, 39 optimizer steps |
| Full: `qlora-r16-lr2e-4-full-e1` | 6,031 retained examples; target 1 epoch / 377 steps |
| Original full-run interruption | Last log: step 40; verified saved checkpoint later found at step 25 |
| November 13 resume | Checkpoint-25 checksums passed; matching run identity; resume message observed; step 27 reached |

Smoke dev execution accuracy: baseline 168/278 (60.43%), adapter 169/278 (60.79%). The paired database-bootstrap interval for the difference was −4.63 to +5.93 percentage points. These provisional scores do not establish an improvement; 26 reference adjudications remained unresolved. Final test evaluation was not reported.

The resumed full run emitted GPU allocation warnings and continued through at least step 27. Completion and later progress are **unknown** in this notebook.

## 1. Local controls

Set `PROJECT` to your checkout. Session names match the conversation. `--timeout 120` means a 120-second CLI wait limit; `--session-hours 2` is the separate training invocation budget. A completed budget pause does not shut down the Colab VM.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import textwrap

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
SESSION = "sql-resume"
RUN_ROOT = "/content/drive/MyDrive/text-to-sql/runs/qlora-r16-lr2e-4-full-e1"

ENABLE_READS = False
ENABLE_SETUP = False
ENABLE_TRAINING = False
ENABLE_EVALUATION = False


def local(args, *, enabled=False, body=None):
    if not enabled:
        print("Skipped:", " ".join(map(str, args)))
        return
    if shutil.which(str(args[0])) is None:
        raise RuntimeError(f"Command unavailable: {args[0]}")
    result = subprocess.run(
        list(map(str, args)), cwd=PROJECT, input=body,
        text=True, capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    result.check_returncode()


def remote(body, *, enabled=False, timeout=120, session=None):
    local(
        ["colab", "exec", "-s", session or SESSION, "--timeout", str(timeout)],
        enabled=enabled, body=textwrap.dedent(body),
    )

print("Project:", PROJECT)
print("Remote session:", SESSION)
print("All remote operations are disabled by default.")

## 2. Session discovery and Drive access (Mac terminal)

These are terminal commands, retained as reference rather than executable Python cells. `sql-check` was used to inspect Drive without requesting a GPU; `sql-resume` was created with a T4 for training. Reuse an active session instead of creating another one.

```bash
colab sessions

# Historical inspection session:
colab new -s sql-check
colab drivemount -s sql-check

# Training session, only if it does not already exist:
colab new -s sql-resume --gpu T4
colab drivemount -s sql-resume
```

Follow the CLI's current authorization prompt in your browser. Do not copy authorization URLs into this notebook. The original `colab sessions` response found no active session; this alone did not determine whether training had completed.

In [ ]:
local(["colab", "sessions"], enabled=ENABLE_READS)

## 3. Inspect saved state and artifacts

This can run against either mounted session. `state=training` can be stale after a VM disappears. Inspect the log and live process together. Historically, `status.json` last reported step 40 and loss 0.199892, while only checkpoint-25 was available.

In [ ]:
remote(f"""
from pathlib import Path
run = Path({RUN_ROOT!r})
for name in ["status.json", "identity.json"]:
    path = run / name
    print(f"=== {{name}} ===")
    print(path.read_text() if path.is_file() else "Not found")
print("=== Run files ===")
for path in sorted(run.iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))
print("=== Checkpoints ===")
for path in sorted((run / "resume").glob("checkpoint-*")):
    print(path.name, "marker:", (path / "COMPLETE.json").is_file(),
          "manifest:", (path / "MANIFEST.sha256.json").is_file())
log = run / "training.log"
if log.is_file():
    print("=== Last 25 log lines ===")
    print("\n".join(log.read_text(errors="replace").splitlines()[-25:]))
""", enabled=ENABLE_READS)

## 4. Upload the project from the Mac

The archive excludes `.git`, weights, and databases. The saved run's `code_commit` was null. Restore the recorded training source and configuration in the next cell. Other modules come from the current checkout: this is **not a complete historical source snapshot**. For future runs, preserve the entire source tree and dependency lock.

Do not repeat setup while a training process is using this project.

In [ ]:
ARCHIVE = "/tmp/text-to-sql-resume.tar.gz"
local([
    "tar", "-czf", ARCHIVE, "src", "scripts", "configs",
    "pyproject.toml", "README.md", "uv.lock",
], enabled=ENABLE_SETUP)
local([
    "colab", "upload", "-s", SESSION, ARCHIVE,
    "/content/text-to-sql-resume.tar.gz",
], enabled=ENABLE_SETUP)

## 5. Restore source/config and verify checkpoint files

This is the recovery procedure used in the conversation. The observed checkpoint-25 passed all listed file checksums. Presence of `COMPLETE.json` alone is insufficient. This cell also checks that no unexpected checkpoint files are present and that the completion marker belongs to the recorded identity.

In [ ]:
remote(f"""
import hashlib
import json
import shutil
import subprocess
from pathlib import Path

run = Path({RUN_ROOT!r})
project = Path("/content/text-to-sql")
project.mkdir(exist_ok=True)
subprocess.run(["tar", "-xzf", "/content/text-to-sql-resume.tar.gz",
                "-C", str(project)], check=True)
shutil.copy2(run / "train_source.py", project / "src/text_to_sql/model/train.py")
shutil.copy2(run / "train_config.yaml", project / "configs/train-full-t4-e1.yaml")

ckpt = run / "resume/checkpoint-25"
marker = json.loads((ckpt / "COMPLETE.json").read_text())
identity = json.loads((run / "identity.json").read_text())
assert marker.get("identity") == identity, "Checkpoint identity mismatch"
manifest = json.loads((ckpt / "MANIFEST.sha256.json").read_text())
actual = {{}}
for path in ckpt.rglob("*"):
    if path.is_file() and path.name not in ("MANIFEST.sha256.json", "COMPLETE.json"):
        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)
        actual[path.relative_to(ckpt).as_posix()] = digest.hexdigest()
assert actual == manifest, "Checkpoint file/checksum mismatch"
print("Checkpoint-25 verification passed")

import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
""", enabled=ENABLE_SETUP, timeout=300)

## 6. Install the recorded training dependencies

Observed environment: PyTorch `2.14.1+cu130`, CUDA `13.0`, Tesla T4, Transformers `5.22.1`, PEFT `0.21.2`, bitsandbytes `0.50.2`, Accelerate `1.15.0`. PyTorch/CUDA already matched in the recovered VM; this command did not explicitly reinstall them. The identity guard below detects tracked dependency changes. It does not establish that every transitive dependency is identical.

In [ ]:
remote("""
import subprocess
import sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-e", "/content/text-to-sql",
    "transformers==5.22.1", "peft==0.21.2",
    "bitsandbytes==0.50.2", "accelerate==1.15.0",
], check=True)
""", enabled=ENABLE_SETUP, timeout=900)

## 7. Resume training as a detached process

**Run once only when no training process is active and compute remains available.** The original project budget was 12 GPU-hours total, with time reserved for validation and final evaluation; total consumption has not been fully compiled. This launch allows up to two hours for this invocation, including initialization, with optimizer-step completion and checkpoint saving possibly extending the deadline.

The guard checks the saved run identity before training can overwrite `identity.json`. The loader verifies a compatible checkpoint. The same-VM local lock prevents duplicate launches through this cell; it does not protect against launches from another VM or another launcher.

`start_new_session=True` with file redirection detaches the process from the CLI request. The Mac may disconnect after launch; Colab VM shutdown can still interrupt training. Regular checkpoints are saved every 25 steps; a budget pause requests an additional checkpoint. The PID is saved locally for monitoring instead of assuming historical PID 1746.

In [ ]:
remote(f"""
import fcntl
import json
import os
import subprocess
import sys
from pathlib import Path

project = Path("/content/text-to-sql")
run = Path({RUN_ROOT!r})
os.chdir(project)
sys.path.insert(0, str(project / "src"))
from text_to_sql.model.train import load_config, run_identity, _dependency_versions
from text_to_sql.model.checkpoints import latest_complete_checkpoint

cfg = load_config(project / "configs/train-full-t4-e1.yaml")
saved = json.loads((run / "identity.json").read_text())
current = run_identity(cfg, _dependency_versions())
assert current == saved, f"Run identity mismatch; not starting: {{current}}"
checkpoint = latest_complete_checkpoint(run / "resume", current)
assert checkpoint is not None, "No verified compatible checkpoint"
assert not (run / "final/COMPLETE.json").exists(), "Final export exists; inspect before restarting"
print("Resume checkpoint:", checkpoint)

lock = open("/content/text-to-sql-full-e1.lock", "a")
fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
with (run / "training.log").open("a", buffering=1) as log:
    process = subprocess.Popen([
        sys.executable, "-u", "-m", "text_to_sql.model.train",
        "--config", "configs/train-full-t4-e1.yaml", "--session-hours", "2",
    ], cwd=project, stdin=subprocess.DEVNULL, stdout=log,
       stderr=subprocess.STDOUT, start_new_session=True,
       pass_fds=(lock.fileno(),))
Path("/content/text-to-sql-full-e1.pid").write_text(str(process.pid))
print("Started PID:", process.pid)
""", enabled=ENABLE_TRAINING, timeout=180)

## 8. Monitor the process, GPU, status, and log

The historical PID was 1746. Prefer the PID file for future launches. Capture subprocess stdout/stderr explicitly: earlier monitoring commands did not display their inherited output through the CLI.

Healthy evidence includes a live training process, `resuming from ... checkpoint-25`, and increasing optimizer steps. GPU utilization alone does not prove useful progress. Memory allocation warnings occurred before steps 26 and 27; the first was followed by progress, while later recovery was not yet confirmed. A fatal `torch.OutOfMemoryError` or terminated process requires investigation. Do not change batch settings inside the existing run without resolving run-identity compatibility.

HF unauthenticated-request and `torch_dtype` deprecation warnings did not themselves establish a training failure. Immediately after resume, progress-bar ETA can be misleading.

In [ ]:
remote(f"""
import subprocess
from pathlib import Path
pid_file = Path("/content/text-to-sql-full-e1.pid")
pid = pid_file.read_text().strip() if pid_file.exists() else "1746"
for command in [["ps", "-p", pid, "-o", "pid,stat,etime,args"], ["nvidia-smi"]]:
    result = subprocess.run(command, capture_output=True, text=True)
    print("Command:", " ".join(command), "exit:", result.returncode)
    print(result.stdout)
    print(result.stderr)
run = Path({RUN_ROOT!r})
print("=== Status (may be stale) ===")
print((run / "status.json").read_text())
print("=== Last 30 log lines ===")
print("\n".join((run / "training.log").read_text(errors="replace").splitlines()[-30:]))
""", enabled=ENABLE_READS)

## 9. After a pause or completion

- `paused`: the target is unfinished. Reassess remaining compute, then reuse the same configuration and the newest verified checkpoint. Do not rerun archive extraction or package installation in a live training process.
- `complete`: verify the final export and evaluate it; do not launch training again.
- `training` with no live process: stale state is possible. Inspect the error log and verified checkpoints.

A two-hour training budget does not release the VM. Manage the idle session separately after confirming the saved checkpoint. No session-deletion command is supplied here because it was not used in the recorded workflow.

### Evaluation commands from repository documentation

These are **reference commands**, not a claim that the resumed full model has been evaluated. Run them on the Colab GPU after completion. The original smoke comparison used the smoke adapter; the full model must use its own final export and a new output directory. Keep the fixed dev policy, finalize unresolved reference decisions before official scoring, and keep final test data closed until model selection is complete.

In [ ]:
remote(f"""
import subprocess
import sys
from pathlib import Path
run = Path({RUN_ROOT!r})
assert (run / "final/COMPLETE.json").is_file(), "No completed final export"
subprocess.run([
    sys.executable, "-u", "-m", "text_to_sql.evaluation.compare_dev",
    "--adapter", str(run / "final"),
    "--data-config", "configs/data.yaml",
    "--output", "/content/drive/MyDrive/text-to-sql/artifacts/full-e1-dev",
], cwd="/content/text-to-sql", check=True)
""", enabled=ENABLE_EVALUATION, timeout=3600)

For an interrupted evaluation, use its existing output directory and add `--resume` after checking identity compatibility. For the preliminary smoke check, the documented command added `--max-examples 20`; removing it selects the eligible fixed dev subset. The smoke training diagnostic added `--train-smoke-config configs/train-smoke-t4.yaml` with a separate output directory and no `--max-examples`.

## Validation and limitations

This notebook is a command walkthrough, not an executed training report. Local validation checks notebook structure, compiles all Python cells, and executes the complete notebook with all external-operation switches disabled. That does not validate Colab execution, Drive access, dependency installation, GPU training, or detached-process survival in a new session.

To use: open this notebook locally in Jupyter/VS Code, confirm the project path, authenticate the Colab CLI, mount Drive in a terminal, and selectively enable the relevant switches. The commands do not include missing earlier setup steps or unobserved final results. Historical summary values come from the experiment log and pasted outputs, not newly fetched artifacts.

Validation performed: notebook schema and Python syntax passed; all 9 code cells executed sequentially in one Python namespace with external operations disabled. Jupyter-kernel execution was unavailable because the local sandbox blocked socket binding. Saved outputs show skipped operations, not remote results. A notebook-viewer visual inspection was not performed.